# Summerize web-based and pdf research papers
This notebook attempts at scraping research papers from the web and summerizes them
I will use closed source chatgpt and open source ollam to create an agent that takes a research paper and summerizes it.
Trafilatura is going to be used instead of BeautifulSoup because it is specifically designed for academic and article content extraction.
Why Trafilatura:
a. It had academic-specific capabilities like:
- Understands academic article structure
- Preserves citations and references
- Maintains section headers
- Handles footnotes

b. It has built in features like:
- Automatically identifies main content
- Extracts metadata (title, authors, date)
- Preserves tables and formatting
- Handles different character encodings

However trafilatura cannot handle pdf format. Also, for sites with strict access controls For very specific formatting requirements

In [ ]:
import sys
#!{sys.executable} -m pip install pdfplumber 

In [ ]:
#imports
import os, io
import requests
#from langchain_community.document_loaders import PyPDFLoader
#from langchain.text_splitter import RecursiveCharacterTextSplitter
from dotenv.main import load_dotenv
from openai import OpenAI
#from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import trafilatura #library designed specifically for scraping academic content, better than beautifulsoup
import pdfplumber
from bs4 import BeautifulSoup

import ollama



In [ ]:
#load the api key in the .env file
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

#Check the key
if not api_key:
    print("No api key found, recreate the env file")
elif not api_key.startswith("sk-proj-"):
    print("An api key found but it doesn't start with sk-proj")
elif api_key.strip() !=api_key:
    print("An api_key was found but it has extra spaces, remove them and resave the .env file")
else:
    print("Api_key found")

In [ ]:
openai = OpenAI()

message = "Hello chatgpt, I am a new user and I will require your help today"

response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages = [{"role":"user","content":message}]
                 )
print(response.choices[0].message.content)

In [ ]:
response = ollama.chat(model=MODEL, messages=messages)
print(response['message']['content'])

In [ ]:
from openai import OpenAI
ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')   
MODEL_llama = "llama3.2"
MODEL_deep = "deepseek-v2"

In [ ]:
url = "https://datascience.codata.org/articles/10.5334/dsj-2025-007"
scraper = Paper_summerizer()
text =scraper.web_scraper(url)
#text

In [ ]:
# A class that can use closed source openai or ollama
class Paper_summerizer:
    def __init__(self):
        self.header = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
    
        
    def summerizer_driver(self,url,ollama = False, open_ai = False):
        '''
        driver function:
        calls the scraper
            returns raw text
        calls gpt or ollama with the text scraped
            returns response with the summary
            and displays it in markdown
        '''
        #call scaraper
        self.url = url
        text = self.web_scraper(url)

        #call ollama or gpt
        if ollama: 
            summary = self.call_ollama(text) #calls call_ollama function
            return self.display_summery(summary)
        elif open_ai:
            summary = self.call_openai(text) #calls call_gpt function
            return self.display_summery(summary)
           
        else:
            print('You need to choose ollama or gpt by putting model=True')
        
            
    def web_scraper(self, url) -> dict:
        #parses the website returning a dict with text and word count
        #self.url = url
        if url.endswith('.pdf'):
            try:
                response = requests.get(url)
                pdf_file = io.BytesIO(response.content)#creates a binary stream in memory and treats it like a file object
                with pdfplumber.open(pdf_file) as pdf:
                    '''
                    pdfplumber a library that extracts text and data from pdfs. it preserves formatting,
                    handles tables, extract images, etc. It is good for academic papers which have complex formatting
                    '''
                    text = '\n'.join(page.extract_text() for page in pdf.pages)
                    return text 
            except Exception as e:
                print(f'Could not download article error {e}')
            
        else:
            try:
                download_url = trafilatura.fetch_url(url)
                if download_url:
                    text = trafilatura.extract(download_url, include_tables = True)
                    #print("text downloaded successfully")
                    return text
                             #"word_count":len(text.split())
            except Exception as e:
                print(f'Could not download article error {e}')
#---------------------
    def call_ollama(self,text):   
           #calls llm_message and returns a summary
           MODEL = 'llama3.2'
           try:
                response = ollama.chat(model=MODEL, messages= self.llm_message(text))
                return response['message']['content']
           except Exception as e:
                #if there is an error in the response then return an error message
                return f'There is an error in gpt response {e}' 
           
    def call_openai(self,text):
            #creates an openai obj
            #calls llm_message for messages and returns a summary
            try:
                openai = self.load_api_key_gpt() #calls the function that loads the api key
                
                #generate a response from gpt using gpt_message function
                response = openai.chat.completions.create(
                model = "gpt-4o-mini",
                messages = self.llm_message(text))
                return response.choices[0].message.content
            except Exception as e:
                #if there is an error in the response then return an error message
                return f'There is an error in gpt response {e}' 
#--------------------   
    def load_api_key_gpt(self):
        #load the api key in the .env file
            load_dotenv(override=True)
            api_key = os.getenv("OPENAI_API_KEY")
            #Check the key is correct
            if not api_key:
                print("No api key found, recreate the env file")
            elif not api_key.startswith("sk-proj-"):
                print("An api key found but it doesn't start with sk-proj")
            elif api_key.strip() !=api_key:
                print("An api_key was found but it has extra spaces, remove them and resave the .env file")
            else:
                print("Api_key found")
            openai = OpenAI()
            return openai
            
    def sys_prompt_gen(self):
         #function that generates a system prompt to summarize research papers
        system_prompt = system_prompt = """
                you are a research assistant focused on creating clear, structured summaries of research papers. \
                research papaer. The summury should not be more than 1000 words and it should be organized in the following sections:\
                1. SUBJECT & OBJECTIVES that includes: 
                - Primary research topic and field of study
                - Key research questions or hypotheses
                - Theoretical framework or background
                
                2. KEY FINDINGS
                - Main experimental/research results
                - Statistical significance where applicable
                - Important data points and trends
                - Supporting evidence for conclusions
                
                3. RESEARCH CHALLENGES: problems like data collection issues and sample size challenges
                
                4. CONCLUSIONS: 
                - Key findings and how these findings relate to the initial research question
                - Real-world applications or impact
                
                5. FUTURE DIRECTIONS
                The summary should include important tables
                """
        return system_prompt

     
    def user_prompt_gen(self,text):
        #function that generates a user prompt that summarizes papers
        user_prompt = "You are looking at a reseach paper, please give a detailed summary it in 1000 words or less. \
            Focus on: subject and objective, key findings, research challenges, conclusion\
            The output should be in markdown\
            If there are tables and graphs, highlight the most important ones and their key insights and it\
            includes important tables"+text
        return user_prompt
        
    def llm_message(self,text):
        return [
            {"role":"system","content":self.sys_prompt_gen()},
            {"role":"user","content":self.user_prompt_gen(text)}
                ]
    
    def display_summery(self,summary):
        
        return display(Markdown(summary))




In [ ]:
#test ollama
url = 'https://datascience.codata.org/articles/10.5334/dsj-2025-007'
summerizer = Paper_summerizer()
summary = summerizer.summerizer_driver(url,ollama=True)
summary


In [ ]:
#test openai
url = 'https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf'
summerizer = Paper_summerizer()
summary = summerizer.summerizer_driver(url,open_ai=True)
summary
